# ESM-2 Embedding Extraction

**Project:** Beyond Cryo — predicting temperature-sensitive residues from sequence (Deep Learning in Genomics, final project).

**Goal:** Run every protein sequence in the dataset through frozen ESM-2 and save the per-residue embeddings to disk. These embeddings are the sole input features to the 1D CNN prediction head in notebook 05.

**Model:** `facebook/esm2_t33_650M_UR50D` (33-layer, 650 M parameters, 1280-dimensional hidden states).

**Inputs:**
- `data/manifest_v3.csv` — index of all 838 pairs (use `manifest_v3_strict.csv` for the 299-pair headline experiment; both use the same embeddings).
- `data/labels/{pair_id}.npz` — contains the `sequence` field (modeled residues for that pair, already aligned to `delta_b`).

**Outputs:**
- `data/embeddings/{pair_id}.npy` — shape `(L, 1280)` float32 array, where `L = len(sequence)` in the corresponding `.npz`.
  Indices align exactly: `embeddings[i]` is the ESM-2 representation of `sequence[i]`, which is labeled by `delta_b[i]`.
- `data/embeddings/P17707_canonical.npy` — ESM-2 embedding of the full AdoMetDC canonical sequence (held-out inference case).

**Design note:** Embeddings are stored per-pair (not per-UniProt) so they align perfectly with the per-pair labels without a separate residue-mapping step. Sequences that are identical across pairs are embedded only once (hash-based cache) and the result copied — so compute is not wasted on duplicates.

**Held-out rule:** AdoMetDC (P17707) is excluded from the training/val/test manifest and is embedded separately at the end of this notebook.

> **Run time:** ~2–5 min on a GPU, several hours on CPU for the full 838-pair dataset. Use a GPU if possible. On Apple Silicon, MPS acceleration is used automatically.
> Run cells top-to-bottom. Re-runs skip already-computed embeddings.

## 1. Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

# --- Model ---
ESM2_MODEL_NAME = "facebook/esm2_t33_650M_UR50D"  # 33 layers, 1280-dim hidden

# --- Paths (relative to notebook location) ---
DATA_DIR      = Path("drive/MyDrive/genomics final project/data").resolve()
LABELS_DIR    = DATA_DIR / "labels"
SEQ_DIR       = DATA_DIR / "sequences"
EMB_DIR       = DATA_DIR / "embeddings"
MANIFEST_PATH = DATA_DIR / "manifest_v3.csv"   # full 838-pair manifest

EMB_DIR.mkdir(parents=True, exist_ok=True)

# --- Inference parameters ---
# ESM-2 was trained with a max of 1024 tokens; BOS + EOS take 2 slots,
# so the maximum input sequence length is 1022 residues per forward pass.
# Longer sequences are handled via a sliding-window strategy (see Cell 5).
MAX_RESIDUES_PER_PASS = 1022
SLIDING_STRIDE        = 511    # 50% overlap — overlapping positions are averaged

# --- Held-out AdoMetDC ---
ADOMETDC_UNIPROT = "P17707"
# DL loop residue positions in the canonical UniProt sequence (1-indexed):
# DL1: 20-28  (persistently disordered — model should score LOW)
# DL2: 164-174 (temperature-sensitive — model should score HIGH)
# DL3: 292-302 (persistently disordered — model should score LOW)
ADOMETDC_LOOPS = {"DL1": (20, 28), "DL2": (164, 174), "DL3": (292, 302)}

print(f"Embeddings directory: {EMB_DIR}")
print(f"Manifest: {MANIFEST_PATH}")

Embeddings directory: /content/drive/MyDrive/genomics final project/data/embeddings
Manifest: /content/drive/MyDrive/genomics final project/data/manifest_v3.csv


## 2. Imports & hardware

In [ ]:
import sys, hashlib, urllib.request
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ---- PyTorch ----------------------------------------------------------------
try:
    import torch
except ImportError:
    print("Installing torch ...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch"])
    import torch

# ---- HuggingFace Transformers -----------------------------------------------
try:
    from transformers import AutoTokenizer, AutoModel
except ImportError:
    print("Installing transformers ...")
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "transformers", "sentencepiece"
    ])
    from transformers import AutoTokenizer, AutoModel

# ---- Device selection -------------------------------------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device: CUDA — {gpu_name} ({vram_gb:.1f} GB VRAM)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Device: MPS (Apple Silicon)")
else:
    DEVICE = torch.device("cpu")
    print("Device: CPU — inference will be slow for large datasets.")
    print("  Tip: consider running on Google Colab (free T4 GPU) or a machine with CUDA.")

Device: CUDA — Tesla T4 (15.6 GB VRAM)


## 3. Load ESM-2 model & tokenizer

First run downloads ~2.5 GB of model weights from HuggingFace and caches them locally. Subsequent runs load from cache instantly.

In [ ]:
print(f"Loading {ESM2_MODEL_NAME} ...")
print("(First run downloads ~2.5 GB — this may take a few minutes.)")

tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL_NAME)
model     = AutoModel.from_pretrained(ESM2_MODEL_NAME)
model.eval()
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel ready: {n_params / 1e6:.0f} M parameters on {DEVICE}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num layers:  {model.config.num_hidden_layers}")

Loading facebook/esm2_t33_650M_UR50D ...
(First run downloads ~2.5 GB — this may take a few minutes.)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model ready: 651 M parameters on cuda
Hidden size: 1280
Num layers:  33


## 4. Embedding function

Two cases:
1. **Short sequence (≤ 1022 residues):** single forward pass.
2. **Long sequence (> 1022 residues):** sliding-window strategy — tiles of `MAX_RESIDUES_PER_PASS` residues with `SLIDING_STRIDE` step; overlapping positions are averaged. This avoids truncating long proteins.

Output shape: `(len(seq), 1280)` float32.

In [ ]:
HIDDEN_DIM = model.config.hidden_size  # 1280 for esm2_t33_650M


def embed_sequence(seq: str) -> np.ndarray:
    """
    Run a protein sequence through frozen ESM-2.

    Returns a (L, 1280) float32 numpy array where L = len(seq).
    BOS and EOS special tokens are stripped from the output.
    """
    L = len(seq)
    assert L > 0, "Empty sequence"

    if L <= MAX_RESIDUES_PER_PASS:
        # --- Single forward pass ---
        inputs = tokenizer(seq, return_tensors="pt", add_special_tokens=True).to(DEVICE)
        with torch.no_grad():
            out = model(**inputs)
        # last_hidden_state: (1, L+2, D)  [0 = BOS, 1..L = residues, L+1 = EOS]
        emb = out.last_hidden_state[0, 1:L + 1, :].cpu().float().numpy()
        return emb

    # --- Sliding window for long sequences ---
    acc = np.zeros((L, HIDDEN_DIM), dtype=np.float32)
    cnt = np.zeros(L, dtype=np.float32)

    start = 0
    while True:
        end = min(start + MAX_RESIDUES_PER_PASS, L)
        chunk = seq[start:end]
        chunk_len = end - start

        inputs = tokenizer(chunk, return_tensors="pt", add_special_tokens=True).to(DEVICE)
        with torch.no_grad():
            out = model(**inputs)
        chunk_emb = out.last_hidden_state[0, 1:chunk_len + 1, :].cpu().float().numpy()

        acc[start:end] += chunk_emb
        cnt[start:end] += 1.0

        if end == L:
            break
        start += SLIDING_STRIDE

    cnt = np.maximum(cnt, 1.0)  # guard against divide-by-zero
    return acc / cnt[:, None]


# Quick smoke test on a tiny sequence
_test = embed_sequence("ACDEFGHIKLMNPQRSTVWY")
assert _test.shape == (20, HIDDEN_DIM), f"Unexpected shape: {_test.shape}"
print(f"embed_sequence smoke test passed — shape {_test.shape} ✓")

embed_sequence smoke test passed — shape (20, 1280) ✓


## 5. Load manifest and collect sequences

For each pair in the manifest we read the `sequence` field from its `.npz` label file — this is the sequence of residues modeled in that structure pair, already in the same order and length as `delta_b` and `mask`. Embedding this sequence directly gives us a (L, 1280) array that can be indexed into without any further residue-mapping.

Pairs with identical sequences are grouped — the embedding is computed once and reused.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Manifest loaded: {len(manifest)} pairs, {manifest['uniprot'].nunique()} unique UniProts")

# Read sequence from each pair's label file
pair_sequences: dict[str, str] = {}   # pair_id -> sequence string
missing_labels: list[str] = []

for _, row in manifest.iterrows():
    pid = row["pair_id"]
    npz_path = LABELS_DIR / f"{pid}.npz"
    if not npz_path.exists():
        missing_labels.append(pid)
        continue
    data = np.load(npz_path, allow_pickle=True)
    seq = str(data["sequence"])   # stored as numpy scalar string
    pair_sequences[pid] = seq

if missing_labels:
    print(f"\n⚠ {len(missing_labels)} pairs have no label file — skipping:")
    for pid in missing_labels[:5]:
        print(f"  {pid}")
    if len(missing_labels) > 5:
        print(f"  ... and {len(missing_labels) - 5} more")

seq_lengths = [len(s) for s in pair_sequences.values()]
print(f"\nPairs with labels: {len(pair_sequences)}")
print(f"Sequence length — min: {min(seq_lengths)}, median: {int(np.median(seq_lengths))}, max: {max(seq_lengths)}")
print(f"Sequences > {MAX_RESIDUES_PER_PASS} (sliding window): "
      f"{sum(l > MAX_RESIDUES_PER_PASS for l in seq_lengths)}")

# Group by sequence hash — identical sequences embedded only once
def seq_hash(seq: str) -> str:
    return hashlib.md5(seq.encode()).hexdigest()[:12]

hash_to_pids: dict[str, list[str]] = {}
for pid, seq in pair_sequences.items():
    h = seq_hash(seq)
    hash_to_pids.setdefault(h, []).append(pid)

unique_seqs = {h: pair_sequences[pids[0]] for h, pids in hash_to_pids.items()}
print(f"\nUnique sequences (after deduplication): {len(unique_seqs)} "
      f"({len(pair_sequences) - len(unique_seqs)} duplicates saved)")

Manifest loaded: 838 pairs, 838 unique UniProts

Pairs with labels: 838
Sequence length — min: 53, median: 264, max: 1231
Sequences > 1022 (sliding window): 1

Unique sequences (after deduplication): 835 (3 duplicates saved)


## 6. Embed all pairs (cached)

Each unique sequence is embedded once; the result is saved for all pairs that share that sequence. Already-computed files are skipped so re-runs are instant.

In [ ]:
# Check which unique sequences still need to be embedded
# (a sequence needs embedding if ANY of its pairs is missing an embedding file)
def needs_embedding(pids: list[str]) -> bool:
    return any(not (EMB_DIR / f"{pid}.npy").exists() for pid in pids)

todo_hashes = [h for h, pids in hash_to_pids.items() if needs_embedding(pids)]
already_done = len(unique_seqs) - len(todo_hashes)
print(f"Unique sequences to embed: {len(todo_hashes)} ({already_done} already cached)")

failed_pairs: list[str] = []

for h in tqdm(todo_hashes, desc="Embedding"):
    pids = hash_to_pids[h]
    seq  = unique_seqs[h]

    try:
        emb = embed_sequence(seq)
        assert emb.shape == (len(seq), HIDDEN_DIM), \
            f"Shape {emb.shape} != ({len(seq)}, {HIDDEN_DIM})"
    except Exception as exc:
        print(f"\n! Embedding failed for {pids[0]} (len={len(seq)}): {exc}")
        failed_pairs.extend(pids)
        continue

    # Save for every pair that shares this sequence
    for pid in pids:
        out_path = EMB_DIR / f"{pid}.npy"
        if not out_path.exists():
            np.save(out_path, emb)

total_written = sum(1 for pid in pair_sequences if (EMB_DIR / f"{pid}.npy").exists())
print(f"\n✓ Embedding files written: {total_written} / {len(pair_sequences)}")
if failed_pairs:
    print(f"✗ Failed pairs: {len(failed_pairs)}")
    for pid in failed_pairs:
        print(f"  {pid}")

Unique sequences to embed: 835 (0 already cached)


Embedding:   0%|          | 0/835 [00:00<?, ?it/s]


✓ Embedding files written: 838 / 838


## 7. Embed AdoMetDC (held-out inference case)

P17707 is excluded from the training/val/test manifest, so it has no pair embedding. We embed the full canonical UniProt sequence here and save it separately as `data/embeddings/P17707_canonical.npy`. This is used in notebook 05 to generate the per-residue prediction plot for DL1/DL2/DL3.

The canonical sequence is read from `data/sequences/P17707.fasta` if present; otherwise fetched from the UniProt REST API.

In [ ]:
adometdc_emb_path = EMB_DIR / f"{ADOMETDC_UNIPROT}_canonical.npy"

def read_fasta(path: Path) -> str:
    lines = path.read_text().splitlines()
    return "".join(l.strip() for l in lines if not l.startswith(">"))

def fetch_uniprot_fasta(uniprot: str) -> str:
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot}.fasta"
    with urllib.request.urlopen(url, timeout=30) as r:
        text = r.read().decode()
    lines = text.splitlines()
    return "".join(l.strip() for l in lines if not l.startswith(">"))

if adometdc_emb_path.exists():
    print(f"P17707 canonical embedding already cached — loading to verify.")
    adometdc_emb = np.load(adometdc_emb_path)
else:
    fasta_path = SEQ_DIR / f"{ADOMETDC_UNIPROT}.fasta"
    if fasta_path.exists():
        adometdc_seq = read_fasta(fasta_path)
        print(f"Read P17707 from data/sequences/: {len(adometdc_seq)} residues")
    else:
        print("P17707 not in data/sequences/ — fetching from UniProt ...")
        adometdc_seq = fetch_uniprot_fasta(ADOMETDC_UNIPROT)
        # Cache locally
        with fasta_path.open("w") as f:
            f.write(f">{ADOMETDC_UNIPROT}\n")
            for i in range(0, len(adometdc_seq), 60):
                f.write(adometdc_seq[i:i + 60] + "\n")
        print(f"  -> {len(adometdc_seq)} residues fetched and saved")

    print(f"Embedding P17707 ({len(adometdc_seq)} residues) ...")
    adometdc_emb = embed_sequence(adometdc_seq)
    np.save(adometdc_emb_path, adometdc_emb)
    print(f"Saved {adometdc_emb_path.name}")

print(f"\nAdoMetDC embedding shape: {adometdc_emb.shape}   (expected: (N_residues, {HIDDEN_DIM}))")

# Quick per-loop L2-norm check (just to confirm the embedding is sensible;
# meaningful differences between loops won't emerge until the head is trained)
print("\nPer-loop mean embedding L2-norm (pre-head, not predictive yet):")
for loop_name, (res_start, res_end) in ADOMETDC_LOOPS.items():
    # UniProt canonical residues are 1-indexed; convert to 0-indexed slice
    sl = adometdc_emb[res_start - 1 : res_end]
    if sl.size == 0:
        print(f"  {loop_name} ({res_start}–{res_end}): *** index out of range ***")
    else:
        print(f"  {loop_name} (residues {res_start}–{res_end}): "
              f"mean L2 = {np.linalg.norm(sl, axis=1).mean():.3f}")

P17707 not in data/sequences/ — fetching from UniProt ...
  -> 334 residues fetched and saved
Embedding P17707 (334 residues) ...
Saved P17707_canonical.npy

AdoMetDC embedding shape: (334, 1280)   (expected: (N_residues, 1280))

Per-loop mean embedding L2-norm (pre-head, not predictive yet):
  DL1 (residues 20–28): mean L2 = 10.234
  DL2 (residues 164–174): mean L2 = 10.120
  DL3 (residues 292–302): mean L2 = 10.106


## 8. Sanity check — shapes and statistics

In [ ]:
# Verify every pair in the manifest has an embedding, and its shape matches the label sequence
shape_ok    = 0
shape_bad   = []
missing_emb = []
all_means   = []
all_stds    = []

for pid, seq in pair_sequences.items():
    emb_path = EMB_DIR / f"{pid}.npy"
    if not emb_path.exists():
        missing_emb.append(pid)
        continue
    emb = np.load(emb_path)
    if emb.shape != (len(seq), HIDDEN_DIM):
        shape_bad.append((pid, emb.shape, len(seq)))
    else:
        shape_ok += 1
        all_means.append(float(emb.mean()))
        all_stds.append(float(emb.std()))

print(f"Pairs checked: {len(pair_sequences)}")
print(f"  Shape correct:   {shape_ok}")
print(f"  Shape mismatch:  {len(shape_bad)}")
print(f"  Missing file:    {len(missing_emb)}")

if shape_bad:
    print("\nShape mismatches (first 5):")
    for pid, got, expected_l in shape_bad[:5]:
        print(f"  {pid}: got {got}, expected ({expected_l}, {HIDDEN_DIM})")

if missing_emb:
    print("\nMissing embeddings (first 5):")
    for pid in missing_emb[:5]:
        print(f"  {pid}")

if all_means:
    print(f"\nEmbedding statistics across all pairs:")
    print(f"  Per-array mean — avg: {np.mean(all_means):.4f}, std: {np.std(all_means):.4f}")
    print(f"  Per-array std  — avg: {np.mean(all_stds):.4f},  std: {np.std(all_stds):.4f}")

# Quick spot-check of one example
print("\nSpot-check — first 3 pairs:")
for pid in list(pair_sequences.keys())[:3]:
    emb_path = EMB_DIR / f"{pid}.npy"
    npz_path = LABELS_DIR / f"{pid}.npz"
    emb  = np.load(emb_path)
    data = np.load(npz_path, allow_pickle=True)
    seq  = str(data["sequence"])
    mask = data["mask"]
    print(f"  {pid}")
    print(f"    emb shape:    {emb.shape}")
    print(f"    seq length:   {len(seq)}")
    print(f"    mask (both modeled): {mask.sum()} / {len(mask)} residues")
    print(f"    emb[:5] L2:   {np.linalg.norm(emb[:5], axis=1).tolist()}")

Pairs checked: 838
  Shape correct:   838
  Shape mismatch:  0
  Missing file:    0

Embedding statistics across all pairs:
  Per-array mean — avg: -0.0007, std: 0.0004
  Per-array std  — avg: 0.2740,  std: 0.0049

Spot-check — first 3 pairs:
  P94692_1KEK_2C3P
    emb shape:    (1231, 1280)
    seq length:   1231
    mask (both modeled): 1231 / 1231 residues
    emb[:5] L2:   [10.167383193969727, 10.075676918029785, 10.058929443359375, 9.960504531860352, 9.88187026977539]
  P22983_2DIK_1DIK
    emb shape:    (869, 1280)
    seq length:   869
    mask (both modeled): 869 / 869 residues
    emb[:5] L2:   [9.939640045166016, 9.952404022216797, 9.861157417297363, 9.946172714233398, 9.893905639648438]
  Q54468_1C7T_1QBA
    emb shape:    (858, 1280)
    seq length:   858
    mask (both modeled): 858 / 858 residues
    emb[:5] L2:   [10.153972625732422, 10.33988094329834, 10.244196891784668, 10.0849609375, 10.0596923828125]


## 9. Build the embedding index for the training notebook

This cell creates a small lookup table (`data/embeddings/index.csv`) mapping each `pair_id` to its embedding file path, split assignment, and sequence length. Notebook 05 (training) reads this to construct DataLoaders without re-scanning the directory.

In [ ]:
index_rows = []
for _, row in manifest.iterrows():
    pid = row["pair_id"]
    emb_path = EMB_DIR / f"{pid}.npy"
    if not emb_path.exists():
        continue
    seq = pair_sequences.get(pid, "")
    index_rows.append({
        "pair_id":        pid,
        "uniprot":        row["uniprot"],
        "split_v3":       row.get("split_v3", ""),
        "seq_cluster_v3": row.get("seq_cluster_v3", ""),
        "ligand_match":   row.get("ligand_match", ""),
        "seq_len":        len(seq),
        "emb_path":       str(emb_path.relative_to(DATA_DIR)),
        "label_path":     f"labels/{pid}.npz",
    })

index_df = pd.DataFrame(index_rows)
index_path = EMB_DIR / "index.csv"
index_df.to_csv(index_path, index=False)

print(f"Embedding index saved: {index_path.name}")
print(f"  Total entries: {len(index_df)}")
if "split_v3" in index_df.columns:
    print("  Split counts (split_v3):")
    for split, cnt in index_df["split_v3"].value_counts().items():
        print(f"    {split}: {cnt}")
    strict = index_df[index_df["ligand_match"] == True] if "ligand_match" in index_df.columns else pd.DataFrame()
    if len(strict):
        print(f"  Strict subset (ligand_match=True): {len(strict)} pairs")
print("\nindex_df.head():")
index_df.head()

Embedding index saved: index.csv
  Total entries: 838
  Split counts (split_v3):
    train: 587
    test: 126
    val: 125
  Strict subset (ligand_match=True): 299 pairs

index_df.head():


,pair_id,uniprot,split_v3,seq_cluster_v3,ligand_match,seq_len,emb_path,label_path
0,P94692_1KEK_2C3P,P94692,test,jacc_0558,False,1231,embeddings/P94692_1KEK_2C3P.npy,labels/P94692_1KEK_2C3P.npz
1,P22983_2DIK_1DIK,P22983,train,jacc_0391,True,869,embeddings/P22983_2DIK_1DIK.npy,labels/P22983_2DIK_1DIK.npz
2,Q54468_1C7T_1QBA,Q54468,val,jacc_0634,True,858,embeddings/Q54468_1C7T_1QBA.npy,labels/Q54468_1C7T_1QBA.npz
3,Q56F26_2VZS_2X09,Q56F26,train,jacc_0638,False,858,embeddings/Q56F26_2VZS_2X09.npy,labels/Q56F26_2VZS_2X09.npz
4,P09186_1ROV_1N8Q,P09186,train,jacc_0261,False,849,embeddings/P09186_1ROV_1N8Q.npy,labels/P09186_1ROV_1N8Q.npz


## Done — next steps

All embeddings are now saved in `data/embeddings/`. Here's what's ready:

```
data/embeddings/
├── index.csv                  ← embedding index for the training notebook
├── P17707_canonical.npy       ← AdoMetDC held-out inference embedding
├── {pair_id}.npy  (x838)      ← per-pair embeddings, shape (L, 1280)
```

**How to load in notebook 05 (training):**

```python
# Load embedding + labels for one pair
emb  = np.load(f"data/embeddings/{pair_id}.npy")      # (L, 1280)
data = np.load(f"data/labels/{pair_id}.npz")           # delta_b, mask, sequence, seq_idx
mask = data["mask"]                                     # boolean, (L,)
y    = data["delta_b"]                                  # float32, (L,)

# Only supervised positions (both structures modeled the residue)
X_sup = emb[mask]   # (n_modeled, 1280)
y_sup = y[mask]     # (n_modeled,)
```

**Split strategy (use split_v3 from index.csv or manifest_v3.csv):**
- `train` → fit CNN head
- `val`   → early stopping / hyperparameter selection
- `test`  → held-out Spearman correlation for the results table
- `P17707_canonical.npy` → independent AdoMetDC case study (not in any split)

**Headline experiment:** train and evaluate on the strict subset (`ligand_match == True`, 299 pairs). Consider the full 838-pair set with mismatched-pair downweighting as an ablation.